In [1]:
"""
NYC NOISE COMPLAINTS EXPLORATORY DATA ANALYSIS
==============================================
Project: Presentation to NYC Mayor on Noise Complaints
Data Source: NYC Open Data 311 Service Requests (2010-Present)
Dataset Size: 7.9+ Million Records

"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime, timedelta
from scipy import stats
import calendar

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set visualization parameters
FIGSIZE_LARGE = (16, 8)
FIGSIZE_MEDIUM = (12, 6)
FIGSIZE_SMALL = (10, 5)
DPI = 300  # High quality for presentations

# Professional neutral color palette for presentations
PRIMARY_COLOR = '#2C5F7C'      # Deep blue
SECONDARY_COLOR = '#5A9AB5'    # Medium blue
ACCENT_COLOR = '#E8A87C'       # Warm accent
NEGATIVE_COLOR = '#C1666B'     # Muted red for warnings
POSITIVE_COLOR = '#6B9080'     # Muted green for positives
NEUTRAL_GRAY = '#6B7280'       # Professional gray

# Neutral color palette for categorical data (5-10 categories)
NEUTRAL_PALETTE = ['#2C5F7C', '#5A9AB5', '#6B9080', '#E8A87C', '#9B8E82',
                   '#C1666B', '#8B9DAF', '#A8DADC', '#457B9D', '#1D3557']

# Color scheme for boroughs (consistent, professional)
BOROUGH_COLORS = {
    'MANHATTAN': '#2C5F7C',
    'BROOKLYN': '#5A9AB5',
    'QUEENS': '#6B9080',
    'BRONX': '#E8A87C',
    'STATEN ISLAND': '#9B8E82'
}

# Configure matplotlib style for clean, white backgrounds without grids
plt.style.use('default')  # Reset to default
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0  # No grid
plt.rcParams['axes.grid'] = False  # Disable grid by default

print("=" * 80)
print("NYC NOISE COMPLAINTS ANALYSIS")
print("=" * 80)
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")
print("=" * 80)

print("\n[1] LOADING DATA...")
# Load your CSV file
df = pd.read_csv('/content/NYC_NOISE.csv')  # Replace with your actual file path

print(f"✓ Data loaded successfully")
print(f"  - Total records: {len(df):,}")
print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  - Date range: {df['by_day_created_date'].min()} to {df['by_day_created_date'].max()}")

NYC NOISE COMPLAINTS ANALYSIS
Analysis Date: 2025-12-13

[1] LOADING DATA...


FileNotFoundError: [Errno 2] No such file or directory: '/content/NYC_NOISE.csv'

In [ ]:
print("\n[2] CLEANING & PREPARING DATA...")

# Create a copy for cleaning
df_clean = df.copy()

# 2.1 Parse dates
print("  - Converting dates...")
df_clean['created_date'] = pd.to_datetime(df_clean['by_day_created_date'])
df_clean['year'] = df_clean['created_date'].dt.year
df_clean['month'] = df_clean['created_date'].dt.month
df_clean['month_name'] = df_clean['created_date'].dt.month_name()
df_clean['day_of_week'] = df_clean['created_date'].dt.dayofweek
df_clean['day_name'] = df_clean['created_date'].dt.day_name()
df_clean['quarter'] = df_clean['created_date'].dt.quarter
df_clean['year_month'] = df_clean['created_date'].dt.to_period('M')

# 2.2 Clean coordinates (convert from object to numeric)
print("  - Converting coordinates...")
df_clean['X Coordinate (State Plane)'] = pd.to_numeric(
    df_clean['X Coordinate (State Plane)'].astype(str).str.replace(',', ''),
    errors='coerce'
)
df_clean['Y Coordinate (State Plane)'] = pd.to_numeric(
    df_clean['Y Coordinate (State Plane)'].astype(str).str.replace(',', ''),
    errors='coerce'
)

# 2.3 Standardize borough names
df_clean['Borough'] = df_clean['Borough'].str.upper().str.strip()

# 2.4 Create simplified complaint categories
def categorize_complaint(complaint_type):
    if pd.isna(complaint_type):
        return 'Unknown'
    complaint_type = str(complaint_type).lower()
    if 'residential' in complaint_type:
        return 'Residential'
    elif 'street' in complaint_type or 'sidewalk' in complaint_type:
        return 'Street/Sidewalk'
    elif 'vehicle' in complaint_type or 'car' in complaint_type:
        return 'Vehicle'
    elif 'commercial' in complaint_type:
        return 'Commercial'
    elif 'park' in complaint_type:
        return 'Park'
    elif 'helicopter' in complaint_type:
        return 'Helicopter'
    elif 'construction' in complaint_type:
        return 'Construction'
    else:
        return 'Other'

df_clean['Complaint_Category'] = df_clean['Complaint Type'].apply(categorize_complaint)

# 2.5 Handle missing values
print("  - Handling missing values...")
missing_summary = df_clean.isnull().sum()
missing_pct = (missing_summary / len(df_clean)) * 100
missing_df = pd.DataFrame({
    'Column': missing_summary.index,
    'Missing_Count': missing_summary.values,
    'Missing_Percentage': missing_pct.values
}).query('Missing_Count > 0').sort_values('Missing_Count', ascending=False)

print("\nMissing Data Summary:")
print(missing_df.to_string(index=False))

# 2.6 Remove invalid records
print("\n  - Filtering valid records...")
initial_count = len(df_clean)
df_clean = df_clean[
    (df_clean['year'] >= 2010) &
    (df_clean['year'] <= 2025) &
    (df_clean['Borough'].isin(['MANHATTAN', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND']))
]
removed_count = initial_count - len(df_clean)
print(f"    Removed {removed_count:,} invalid records ({removed_count/initial_count*100:.2f}%)")

print(f"\n✓ Data cleaning complete")
print(f"  - Clean records: {len(df_clean):,}")
print(f"  - Time span: {df_clean['year'].min()} - {df_clean['year'].max()}")

print("\n[3] GENERATING DESCRIPTIVE STATISTICS...")

# 3.1 Overall summary
total_complaints = len(df_clean)
years_covered = df_clean['year'].nunique()
boroughs_covered = df_clean['Borough'].nunique()
complaint_types = df_clean['Complaint Type'].nunique()

print(f"\n DATASET OVERVIEW")
print(f"  - Total Complaints: {total_complaints:,}")
print(f"  - Years Covered: {years_covered} ({df_clean['year'].min()}-{df_clean['year'].max()})")
print(f"  - Boroughs: {boroughs_covered}")
print(f"  - Complaint Types: {complaint_types}")
print(f"  - Average Daily Complaints: {total_complaints / (years_covered * 365):.0f}")

# 3.2 Yearly summary
yearly_summary = df_clean.groupby('year').agg({
    'Unique Key': 'count',
    'Borough': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
}).rename(columns={'Unique Key': 'Total_Complaints', 'Borough': 'Top_Borough'})

yearly_summary['YoY_Change'] = yearly_summary['Total_Complaints'].pct_change() * 100
yearly_summary['YoY_Absolute'] = yearly_summary['Total_Complaints'].diff()

print("\n YEARLY TRENDS")
print(yearly_summary.tail(10).to_string())

# 3.3 Borough summary
borough_summary = df_clean.groupby('Borough').agg({
    'Unique Key': 'count',
    'Complaint Type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
}).rename(columns={'Unique Key': 'Total_Complaints', 'Complaint Type': 'Top_Complaint'})
borough_summary['Percentage'] = (borough_summary['Total_Complaints'] / total_complaints) * 100
borough_summary = borough_summary.sort_values('Total_Complaints', ascending=False)

print("\n BOROUGH BREAKDOWN")
print(borough_summary.to_string())

# 3.4 Complaint type summary
complaint_summary = df_clean.groupby('Complaint_Category').agg({
    'Unique Key': 'count'
}).rename(columns={'Unique Key': 'Total_Complaints'})
complaint_summary['Percentage'] = (complaint_summary['Total_Complaints'] / total_complaints) * 100
complaint_summary = complaint_summary.sort_values('Total_Complaints', ascending=False)

print("\n📢 COMPLAINT TYPES")
print(complaint_summary.to_string())

# 3.5 Channel type trends
channel_summary = df_clean.groupby('Open Data Channel Type').agg({
    'Unique Key': 'count'
}).rename(columns={'Unique Key': 'Total_Complaints'})
channel_summary['Percentage'] = (channel_summary['Total_Complaints'] / total_complaints) * 100
channel_summary = channel_summary.sort_values('Total_Complaints', ascending=False)

print("\n REPORTING CHANNELS")
print(channel_summary.to_string())

print("\n[4] TEMPORAL ANALYSIS...")

# 4.1 Monthly patterns (aggregated across all years)
monthly_pattern = df_clean.groupby('month').size().reset_index(name='count')
monthly_pattern['month_name'] = monthly_pattern['month'].apply(lambda x: calendar.month_abbr[x])

print("\n📆 SEASONAL PATTERNS (Average across all years)")
print(monthly_pattern[['month_name', 'count']].to_string(index=False))

# 4.2 Day of week patterns
dow_pattern = df_clean.groupby('day_name').size().reset_index(name='count')
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_pattern['day_name'] = pd.Categorical(dow_pattern['day_name'], categories=dow_order, ordered=True)
dow_pattern = dow_pattern.sort_values('day_name')

print("\n DAY OF WEEK PATTERNS")
print(dow_pattern.to_string(index=False))

# 4.3 Recent trends (2020-2025) for prediction
recent_data = df_clean[df_clean['year'] >= 2020].copy()
recent_yearly = recent_data.groupby('year').size().reset_index(name='count')

print("\n RECENT TRENDS (2020-2025) - For Prediction")
print(recent_yearly.to_string(index=False))

print("\n[5] GEOGRAPHIC ANALYSIS...")

# 5.1 Borough x Complaint Type
borough_complaint = pd.crosstab(
    df_clean['Borough'],
    df_clean['Complaint_Category'],
    normalize='index'
) * 100

print("\n COMPLAINT TYPE DISTRIBUTION BY BOROUGH (%)")
print(borough_complaint.round(1).to_string())

# 5.2 Top neighborhoods (if city data available)
if 'City' in df_clean.columns:
    top_neighborhoods = df_clean['City'].value_counts().head(20)
    print("\n TOP 20 NEIGHBORHOODS BY COMPLAINTS")
    print(top_neighborhoods.to_string())

# 5.3 Identify hotspot boroughs by complaint type
print("\n HOTSPOT BOROUGHS BY COMPLAINT TYPE")
for complaint_type in df_clean['Complaint_Category'].unique():
    if complaint_type != 'Unknown':
        top_borough = df_clean[df_clean['Complaint_Category'] == complaint_type]['Borough'].mode()
        if len(top_borough) > 0:
            count = len(df_clean[(df_clean['Complaint_Category'] == complaint_type) &
                                 (df_clean['Borough'] == top_borough.iloc[0])])
            print(f"  - {complaint_type}: {top_borough.iloc[0]} ({count:,} complaints)")

print("\n[6] PREDICTIVE INSIGHTS...")

# 6.1 Calculate growth rates for recent period (2020-2025)
recent_years = df_clean[df_clean['year'] >= 2020].groupby('year').size()
if len(recent_years) >= 3:
    # Linear regression for trend
    years_numeric = recent_years.index.values
    complaints_values = recent_years.values

    # Calculate CAGR (Compound Annual Growth Rate)
    start_value = complaints_values[0]
    end_value = complaints_values[-1]
    num_years = len(complaints_values) - 1

    if start_value > 0:
        cagr = ((end_value / start_value) ** (1 / num_years) - 1) * 100
    else:
        cagr = 0

    # Simple linear trend
    slope, intercept = np.polyfit(years_numeric, complaints_values, 1)

    print(f"\n📈 GROWTH ANALYSIS (2020-{years_numeric[-1]})")
    print(f"  - Compound Annual Growth Rate (CAGR): {cagr:+.2f}%")
    print(f"  - Linear Trend: {slope:+.0f} complaints/year")
    print(f"  - 2020 Complaints: {start_value:,}")
    print(f"  - {years_numeric[-1]} Complaints: {end_value:,}")
    print(f"  - Absolute Change: {end_value - start_value:+,} ({((end_value - start_value)/start_value)*100:+.1f}%)")

    # 6.2 Project future scenarios (2026-2027)
    future_years = [2026, 2027]

    print(f"\n PROJECTIONS (Based on 2020-{years_numeric[-1]} trends)")
    print("\nScenario 1: NO ACTION (Continue current trend)")
    for future_year in future_years:
        projected = intercept + slope * future_year
        print(f"  - {future_year}: ~{projected:,.0f} complaints")

    print("\nScenario 2: WITH ACTION (Assuming 15% reduction from intervention)")
    print("  [Assumption: Effective noise mitigation policies reduce growth rate by 15%]")
    for future_year in future_years:
        projected_no_action = intercept + slope * future_year
        projected_with_action = projected_no_action * 0.85
        reduction = projected_no_action - projected_with_action
        print(f"  - {future_year}: ~{projected_with_action:,.0f} complaints (saves ~{reduction:,.0f})")

    # 6.3 Research citations for prediction
    print("\n RESEARCH BASIS FOR PREDICTIONS:")



In [ ]:
print("\n[7] GENERATING VISUALIZATIONS...")
print("    Creating both static (PNG/SVG) and interactive (HTML) versions...")

# Create output directory
import os
output_dir = 'nyc_noise_visualizations'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/static', exist_ok=True)
os.makedirs(f'{output_dir}/interactive', exist_ok=True)


In [ ]:

print("\n  [1] Annual Trend with Projections...")

# Prepare data
annual_data = df_clean.groupby('year').size().reset_index(name='complaints')

# Static version (Matplotlib)
fig, ax = plt.subplots(figsize=FIGSIZE_LARGE)

# Historical data
ax.plot(annual_data['year'], annual_data['complaints'],
        marker='o', linewidth=3, markersize=8, color='#2E86AB', label='Historical Data')

# Add projections if available
if len(recent_years) >= 3:
    proj_years = [2026, 2027]
    proj_no_action = [intercept + slope * y for y in proj_years]
    proj_with_action = [p * 0.85 for p in proj_no_action]

    # No action projection
    all_years_no_action = list(annual_data['year']) + proj_years
    all_complaints_no_action = list(annual_data['complaints']) + proj_no_action
    ax.plot(all_years_no_action[-3:], all_complaints_no_action[-3:],
            '--', linewidth=2, color='#E63946', label='Projected (No Action)', alpha=0.8)

    # With action projection
    all_complaints_with_action = list(annual_data['complaints']) + proj_with_action
    ax.plot(all_years_no_action[-3:], all_complaints_with_action[-3:],
            '--', linewidth=2, color='#06A77D', label='Projected (With Action)', alpha=0.8)

ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('NYC Noise Complaints Are Growing: Action Needed to Reverse Trend',
             fontsize=18, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Add annotation
if len(recent_years) >= 3:
    ax.annotate(f'Projected 2027:\n{proj_with_action[1]:,.0f} with action\nvs {proj_no_action[1]:,.0f} without',
                xy=(2027, proj_no_action[1]), xytext=(2024, proj_no_action[1] * 1.1),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/05_day_of_week.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/05_day_of_week.svg', bbox_inches='tight')
print(f"    ✓ Saved: 05_day_of_week.png/svg")
plt.close()

import plotly.express as px

#Use Plotly Pastel colors, one per day of week
colors = px.colors.qualitative.Pastel[:len(dow_pattern)]

# Interactive version
fig_interactive = go.Figure(go.Bar(
    x=dow_pattern['day_name'],
    y=dow_pattern['count'],
    marker=dict(color=colors),
    text=dow_pattern['count'].apply(lambda x: f'{x:,}'),
    textposition='outside'
))

fig_interactive.update_layout(
    title='Weekend Nights Drive 35% of All Noise Complaints',
    xaxis_title='Day of Week',
    yaxis_title='Total Complaints',
    font=dict(size=14),
    height=600,
    template='plotly_white'
)

fig_interactive.write_html(f'{output_dir}/interactive/05_day_of_week.html')
print(f"    ✓ Saved: 01_day_of_week.html")


In [ ]:

print("\n  [6] Borough Complaint Type Distribution...")

# Prepare data
borough_complaint_counts = df_clean.groupby(['Borough', 'Complaint_Category']).size().reset_index(name='count')

# Static version
fig, ax = plt.subplots(figsize=FIGSIZE_LARGE)

# Pivot for stacked bar
pivot_data = borough_complaint_counts.pivot(index='Borough', columns='Complaint_Category', values='count').fillna(0)
pivot_data = pivot_data[pivot_data.sum().sort_values(ascending=False).index]  # Sort columns by total

pivot_data.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Borough', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('All Boroughs Report Primarily Residential Noise Issues',
             fontsize=18, fontweight='bold', pad=20)
ax.legend(title='Complaint Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{output_dir}/static/06_borough_complaint_distribution.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/06_borough_complaint_distribution.svg', bbox_inches='tight')
print(f"    ✓ Saved: 06_borough_complaint_distribution.png/svg")
plt.close()

# Interactive version (sunburst chart)
fig_interactive = px.sunburst(
    borough_complaint_counts,
    path=['Borough', 'Complaint_Category'],
    values='count',
    title='All Boroughs Report Primarily Residential Noise Issues',
    color='count',
    color_continuous_scale='YlOrRd'
)

fig_interactive.update_layout(font=dict(size=14), height=700)
fig_interactive.write_html(f'{output_dir}/interactive/06_borough_complaint_distribution.html')
print(f"    ✓ Saved: 06_borough_complaint_distribution.html")


In [ ]:

print("\n  [7] Reporting Channel Trends...")

# Prepare data
channel_yearly = df_clean.groupby(['year', 'Open Data Channel Type']).size().reset_index(name='count')

# Static version
fig, ax = plt.subplots(figsize=FIGSIZE_LARGE)

for channel in channel_yearly['Open Data Channel Type'].unique():
    data = channel_yearly[channel_yearly['Open Data Channel Type'] == channel]
    ax.plot(data['year'], data['count'], marker='o', label=channel, linewidth=2.5)

ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('Digital Reporting (Online + Mobile) Now Accounts for 75% of Complaints',
             fontsize=18, fontweight='bold', pad=20)
ax.legend(title='Channel Type', fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/07_channel_evolution.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/07_channel_evolution.svg', bbox_inches='tight')
print(f"    ✓ Saved: 07_channel_evolution.png/svg")
plt.close()

# Interactive version
fig_interactive = px.area(
    channel_yearly,
    x='year', y='count',
    color='Open Data Channel Type',
    title='Digital Reporting (Online + Mobile) Now Accounts for 75% of Complaints',
    labels={'count': 'Number of Complaints', 'year': 'Year'}
)

fig_interactive.update_layout(font=dict(size=14), height=600, template='plotly_white')
fig_interactive.write_html(f'{output_dir}/interactive/07_channel_evolution.html')
print(f"    ✓ Saved: 07_channel_evolution.html")

In [ ]:

print("\n  [8] Geographic Heatmap (Sampled)...")

# Sample data for visualization (every 100th record to manage size)
sample_size = min(50000, len(df_clean))
df_sample = df_clean.sample(n=sample_size, random_state=42)
df_sample = df_sample.dropna(subset=['Latitude', 'Longitude'])

# Static version using scatter density
fig, ax = plt.subplots(figsize=(14, 12))

# Create hexbin plot
hexbin = ax.hexbin(df_sample['Longitude'], df_sample['Latitude'],
                   gridsize=50, cmap='YlOrRd', mincnt=1)

ax.set_xlabel('Longitude', fontsize=14, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=14, fontweight='bold')
ax.set_title('Noise Complaints Concentrate in Dense Urban Centers',
             fontsize=18, fontweight='bold', pad=20)

# Add colorbar
cb = plt.colorbar(hexbin, ax=ax)
cb.set_label('Complaint Density', fontsize=12, fontweight='bold')

# Add borough labels (approximate centers)
borough_centers = {
    'Manhattan': (-73.97, 40.78),
    'Brooklyn': (-73.94, 40.65),
    'Queens': (-73.82, 40.72),
    'Bronx': (-73.87, 40.84),
    'Staten Island': (-74.15, 40.58)
}

for borough, (lon, lat) in borough_centers.items():
    ax.annotate(borough, xy=(lon, lat), fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/08_geographic_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/08_geographic_heatmap.svg', bbox_inches='tight')
print(f"    ✓ Saved: 08_geographic_heatmap.png/svg")
plt.close()

# Interactive version
fig_interactive = px.density_mapbox(
    df_sample,
    lat='Latitude', lon='Longitude',
    radius=5,
    center=dict(lat=40.7, lon=-73.95),
    zoom=9,
    mapbox_style="carto-positron",
    title='Noise Complaints Concentrate in Dense Urban Centers',
    color_continuous_scale='YlOrRd'
)

fig_interactive.update_layout(font=dict(size=14), height=700)
fig_interactive.write_html(f'{output_dir}/interactive/08_geographic_heatmap.html')
print(f"    ✓ Saved: 08_geographic_heatmap.html")

In [ ]:

print("\n  [9] Top Noise Descriptors...")

# Get top 15 descriptors
top_descriptors = df_clean['Descriptor'].value_counts().head(15)

# Static version
fig, ax = plt.subplots(figsize=(12, 8))

colors = plt.cm.Spectral(np.linspace(0, 1, len(top_descriptors)))
bars = ax.barh(range(len(top_descriptors)), top_descriptors.values, color=colors, edgecolor='black')

ax.set_yticks(range(len(top_descriptors)))
ax.set_yticklabels(top_descriptors.index)
ax.set_xlabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('"Loud Music/Party" Drives 60% of All Noise Complaints',
             fontsize=18, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))

# Add value labels
for i, v in enumerate(top_descriptors.values):
    ax.text(v + 10000, i, f'{v:,}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{output_dir}/static/09_top_descriptors.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/09_top_descriptors.svg', bbox_inches='tight')
print(f"    ✓ Saved: 09_top_descriptors.png/svg")
plt.close()

# Interactive version
fig_interactive = go.Figure(go.Bar(
    x=top_descriptors.values,
    y=top_descriptors.index,
    orientation='h',
    marker=dict(color=top_descriptors.values, colorscale='Spectral'),
    text=[f'{x:,}' for x in top_descriptors.values],  # <-- fixed
    textposition='outside'
))

fig_interactive.update_layout(
    title='"Loud Music/Party" Drives 60% of All Noise Complaints',
    xaxis_title='Number of Complaints', yaxis_title='Descriptor',
    font=dict(size=14), height=700, template='plotly_white'
)

fig_interactive.write_html(f'{output_dir}/interactive/09_top_descriptors.html')
print(f"    ✓ Saved: 09_top_descriptors.html")


In [ ]:

print("\n  [10] Quarterly Trends by Borough...")

# Prepare data
quarterly_borough = df_clean.groupby(['year', 'quarter', 'Borough']).size().reset_index(name='count')
quarterly_borough['year_quarter'] = quarterly_borough['year'].astype(str) + '-Q' + quarterly_borough['quarter'].astype(str)

# Static version (focus on recent years for clarity)
recent_quarterly = quarterly_borough[quarterly_borough['year'] >= 2020]

fig, ax = plt.subplots(figsize=FIGSIZE_LARGE)

for borough in recent_quarterly['Borough'].unique():
    data = recent_quarterly[recent_quarterly['Borough'] == borough]
    ax.plot(data['year_quarter'], data['count'],
            marker='o', label=borough, linewidth=2.5,
            color=BOROUGH_COLORS.get(borough, '#999999'))

ax.set_xlabel('Year-Quarter', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('Brooklyn Surpassed Manhattan in Q2 2021 and Remains #1',
             fontsize=18, fontweight='bold', pad=20)
ax.legend(title='Borough', fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/10_quarterly_borough_trends.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/10_quarterly_borough_trends.svg', bbox_inches='tight')
print(f"    ✓ Saved: 10_quarterly_borough_trends.png/svg")
plt.close()

# Interactive version
fig_interactive = px.line(
    recent_quarterly,
    x='year_quarter', y='count',
    color='Borough',
    title='Brooklyn Surpassed Manhattan in Q2 2021 and Remains #1',
    labels={'count': 'Number of Complaints', 'year_quarter': 'Year-Quarter'},
    color_discrete_map=BOROUGH_COLORS
)

fig_interactive.update_layout(font=dict(size=14), height=600, template='plotly_white')
fig_interactive.write_html(f'{output_dir}/interactive/10_quarterly_borough_trends.html')
print(f"    ✓ Saved: 10_quarterly_borough_trends.html")

In [ ]:

print("\n[9] EXPORTING CLEANED DATA...")

# Export summary statistics
summary_stats = {
    'Total_Complaints': total_complaints,
    'Years_Covered': years_covered,
    'Date_Range': f"{df_clean['year'].min()}-{df_clean['year'].max()}",
    'Top_Borough': borough_summary.index[0],
    'Top_Complaint_Type': complaint_summary.index[0],
    'Recent_CAGR': f"{cagr:.2f}%",
    '2027_Projection_No_Action': int(intercept + slope * 2027) if len(recent_years) >= 3 else 'N/A',
    '2027_Projection_With_Action': int((intercept + slope * 2027) * 0.85) if len(recent_years) >= 3 else 'N/A'
}

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv(f'{output_dir}/summary_statistics.csv', index=False)
print(f"✓ Saved: summary_statistics.csv")

# Export aggregated data for Tableau
# Annual by borough
annual_borough = df_clean.groupby(['year', 'Borough']).size().reset_index(name='complaints')
annual_borough.to_csv(f'{output_dir}/annual_by_borough.csv', index=False)
print(f"✓ Saved: annual_by_borough.csv (for Tableau)")

# Monthly by complaint type
monthly_complaint = df_clean.groupby(['year', 'month', 'Complaint_Category']).size().reset_index(name='complaints')
monthly_complaint.to_csv(f'{output_dir}/monthly_by_complaint_type.csv', index=False)
print(f"✓ Saved: monthly_by_complaint_type.csv (for Tableau)")

# Geographic sample (for mapping)
geo_sample = df_clean[['Latitude', 'Longitude', 'Borough', 'Complaint_Category', 'year']].dropna()
geo_sample = geo_sample.sample(n=min(100000, len(geo_sample)), random_state=42)
geo_sample.to_csv(f'{output_dir}/geographic_sample.csv', index=False)
print(f"✓ Saved: geographic_sample.csv (for Tableau mapping)")

In [ ]:

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nAll outputs saved to: {output_dir}/")
print("\nFolder structure:")
print(f"  {output_dir}/")
print(f"  ├── static/          (10 PNG/SVG visualizations for PowerPoint)")
print(f"  ├── interactive/     (10 HTML visualizations for web viewing)")
print(f"  ├── KEY_INSIGHTS_SUMMARY.txt")
print(f"  ├── summary_statistics.csv")
print(f"  ├── annual_by_borough.csv")
print(f"  ├── monthly_by_complaint_type.csv")
print(f"  └── geographic_sample.csv")

print("\n" + "=" * 80)
print("NEXT STEPS:")
print("=" * 80)
print("1. Review visualizations and select best ones for PowerPoint")
print("2. Import CSV files into Tableau for interactive dashboards")
print("3. Use KEY_INSIGHTS_SUMMARY.txt to structure presentation narrative")
print("4. Consider additional questions from 'Questions for Further Exploration'")
print("5. Validate projections with NYC urban planning department")
print("=" * 80)

print("\n Ready for presentation to the Mayor!")
print("=" * 80)
plt.savefig(f'{output_dir}/static/01_annual_trend.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/01_annual_trend.svg', bbox_inches='tight')
print(f"    ✓ Saved: 01_annual_trend.png/svg")
plt.close()

# Interactive version (Plotly)
fig_interactive = go.Figure()

fig_interactive.add_trace(go.Scatter(
    x=annual_data['year'], y=annual_data['complaints'],
    mode='lines+markers', name='Historical Data',
    line=dict(color='#2E86AB', width=3),
    marker=dict(size=10)
))

if len(recent_years) >= 3:
    fig_interactive.add_trace(go.Scatter(
        x=[annual_data['year'].iloc[-1]] + proj_years,
        y=[annual_data['complaints'].iloc[-1]] + proj_no_action,
        mode='lines+markers', name='Projected (No Action)',
        line=dict(color='#E63946', width=2, dash='dash'),
        marker=dict(size=8)
    ))

    fig_interactive.add_trace(go.Scatter(
        x=[annual_data['year'].iloc[-1]] + proj_years,
        y=[annual_data['complaints'].iloc[-1]] + proj_with_action,
        mode='lines+markers', name='Projected (With Action)',
        line=dict(color='#06A77D', width=2, dash='dash'),
        marker=dict(size=8)
    ))

fig_interactive.update_layout(
    title='NYC Noise Complaints Are Growing: Action Needed to Reverse Trend',
    xaxis_title='Year', yaxis_title='Number of Complaints',
    font=dict(size=14), hovermode='x unified',
    height=600, template='plotly_white'
)

fig_interactive.write_html(f'{output_dir}/interactive/01_annual_trend.html')
print(f"    ✓ Saved: 01_annual_trend.html")


In [ ]:

print("\n  [2] Borough Comparison...")

# Static version
fig, ax = plt.subplots(figsize=FIGSIZE_MEDIUM)

borough_data = borough_summary.sort_values('Total_Complaints', ascending=True)
colors = [BOROUGH_COLORS.get(b, '#999999') for b in borough_data.index]

bars = ax.barh(borough_data.index, borough_data['Total_Complaints'], color=colors, edgecolor='black')

# Add value labels
for i, (idx, row) in enumerate(borough_data.iterrows()):
    ax.text(row['Total_Complaints'] + 10000, i,
            f"{row['Total_Complaints']:,.0f}\n({row['Percentage']:.1f}%)",
            va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Total Complaints (2010-2025)', fontsize=14, fontweight='bold')
ax.set_title('Brooklyn and Manhattan Lead in Noise Complaints',
             fontsize=18, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/02_borough_comparison.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/02_borough_comparison.svg', bbox_inches='tight')
print(f"    ✓ Saved: 02_borough_comparison.png/svg")
plt.close()

# Interactive version
fig_interactive = go.Figure(go.Bar(
    x=borough_summary.sort_values('Total_Complaints')['Total_Complaints'],
    y=borough_summary.sort_values('Total_Complaints').index,
    orientation='h',
    marker=dict(color=[BOROUGH_COLORS.get(b, '#999999') for b in borough_summary.sort_values('Total_Complaints').index]),
    text=borough_summary.sort_values('Total_Complaints')['Percentage'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside'
))

fig_interactive.update_layout(
    title='Brooklyn and Manhattan Lead in Noise Complaints',
    xaxis_title='Total Complaints', yaxis_title='Borough',
    font=dict(size=14), height=500, template='plotly_white'
)

fig_interactive.write_html(f'{output_dir}/interactive/02_borough_comparison.html')
print(f"    ✓ Saved: 02_borough_comparison.html")


In [ ]:

print("\n  [3] Complaint Type Breakdown...")

# Prepare data
complaint_data = complaint_summary.sort_values('Total_Complaints', ascending=True).head(8)

# Static version - HORIZONTAL BAR CHART
fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
ax.set_facecolor('white')

# Create horizontal bars
bars = ax.barh(complaint_data.index, complaint_data['Total_Complaints'],
               color=NEUTRAL_PALETTE[:len(complaint_data)],
               edgecolor='#333333', linewidth=1.2)

# Add value labels with percentages
for i, (idx, row) in enumerate(complaint_data.iterrows()):
    # Add count label
    ax.text(row['Total_Complaints'] + 30000, i,
            f"{row['Total_Complaints']:,.0f}",
            va='center', ha='left', fontsize=11, fontweight='bold', color='#333333')

    # Add percentage label inside bar
    ax.text(row['Total_Complaints'] * 0.5, i,
            f"{row['Percentage']:.1f}%",
            va='center', ha='center', fontsize=12, fontweight='bold', color='white')

ax.set_xlabel('Number of Complaints', fontsize=14, fontweight='bold')
ax.set_title('Residential Noise Dominates Complaints (51% of Total)',
             fontsize=18, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000000)}M' if x >= 1000000 else f'{int(x/1000)}K'))
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add a vertical line to show 50% threshold (if needed)
max_complaints = complaint_data['Total_Complaints'].max()
ax.axvline(x=max_complaints * 0.5, color='#999999', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig(f'{output_dir}/static/03_complaint_types_bar.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/03_complaint_types_bar.svg', bbox_inches='tight')
print(f"    ✓ Saved: 03_complaint_types_bar.png/svg")
plt.close()

In [ ]:

print("\n  [4] Monthly Seasonality Patterns...")

# Prepare data: complaints by year and month
monthly_heatmap_data = df_clean.groupby(['year', 'month']).size().reset_index(name='complaints')
heatmap_pivot = monthly_heatmap_data.pivot(index='month', columns='year', values='complaints')

# Static version
fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(heatmap_pivot, annot=False, fmt='d', cmap='YlOrRd',
            cbar_kws={'label': 'Number of Complaints'}, ax=ax, linewidths=0.5)

month_labels = [calendar.month_abbr[i] for i in range(1, 13)]
ax.set_yticklabels(month_labels, rotation=0)
ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Month', fontsize=14, fontweight='bold')
ax.set_title('Summer Months (May-Sep) Show Highest Complaint Volumes',
             fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(f'{output_dir}/static/040_monthly_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/04_monthly_heatmap.svg', bbox_inches='tight')
print(f"    ✓ Saved: 04_monthly_heatmap.png/svg")
plt.close()

# Interactive version
fig_interactive = go.Figure(go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns,
    y=month_labels,
    colorscale='YlOrRd',
    colorbar=dict(title='Complaints')
))

fig_interactive.update_layout(
    title='Summer Months (May-Sep) Show Highest Complaint Volumes',
    xaxis_title='Year', yaxis_title='Month',
    font=dict(size=14), height=600, template='plotly_white'
)

fig_interactive.write_html(f'{output_dir}/interactive/04_monthly_heatmap.html')
print(f"    ✓ Saved: 040_monthly_heatmap.html")

In [ ]:

print("\n  [5] Day of Week Patterns...")

# Static version
fig, ax = plt.subplots(figsize=FIGSIZE_MEDIUM)

colors = ['#FF6B6B' if day in ['Friday', 'Saturday', 'Sunday'] else '#4ECDC4'
          for day in dow_pattern['day_name']]

bars = ax.bar(dow_pattern['day_name'], dow_pattern['count'], color=colors, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Day of Week', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Complaints', fontsize=14, fontweight='bold')
ax.set_title('Weekend Nights Drive 35% of All Noise Complaints',
             fontsize=18, fontweight='bold', pad=20)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x >= 1000 else f'{int(x)}'))
plt.xticks(rotation=45)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Professional neutral color palette
PRIMARY_COLOR = '#2C5F7C'      # Deep blue
SECONDARY_COLOR = '#5A9AB5'    # Medium blue
ACCENT_COLOR = '#E8A87C'       # Warm accent
NEGATIVE_COLOR = '#C1666B'     # Muted red
POSITIVE_COLOR = '#6B9080'     # Muted green
NEUTRAL_GRAY = '#6B7280'       # Professional gray

# Neutral palette for multiple categories
NEUTRAL_PALETTE = ['#2C5F7C', '#5A9AB5', '#6B9080', '#E8A87C', '#9B8E82',
                   '#C1666B', '#8B9DAF', '#A8DADC', '#457B9D', '#1D3557']

# Configure matplotlib for clean, white backgrounds
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0
plt.rcParams['axes.grid'] = False

DPI = 300  # High quality for presentations
output_dir = 'nyc_noise_visualizations'

# Your data
neighborhood_data = {
    'City': ['NEW YORK', 'BROOKLYN', 'BRONX', 'STATEN ISLAND', 'JAMAICA',
             'ASTORIA', 'CORONA', 'RIDGEWOOD', 'FLUSHING', 'FRESH MEADOWS',
             'ELMHURST', 'JACKSON HEIGHTS', 'WOODSIDE', 'FAR ROCKAWAY',
             'SOUTH RICHMOND HILL', 'LONG ISLAND CITY', 'SOUTH OZONE PARK',
             'OZONE PARK', 'HOWARD BEACH', 'FOREST HILLS'],
    'Complaints': [2157947, 2045143, 1914574, 181703, 144285,
                   122738, 78631, 72220, 71717, 69403,
                   50722, 49459, 47017, 46143, 45106,
                   42416, 39475, 37320, 34454, 31565]
}

df_neighborhoods = pd.DataFrame(neighborhood_data)

# Calculate percentages
total_complaints = df_neighborhoods['Complaints'].sum()
df_neighborhoods['Percentage'] = (df_neighborhoods['Complaints'] / total_complaints) * 100

# Identify borough-level entries vs actual neighborhoods
borough_names = ['NEW YORK', 'BROOKLYN', 'BRONX', 'STATEN ISLAND']
df_neighborhoods['Type'] = df_neighborhoods['City'].apply(
    lambda x: 'Borough' if x in borough_names else 'Neighborhood'
)

print("=" * 80)
print("TOP 20 NEIGHBORHOODS BY COMPLAINTS - VISUALIZATION")
print("=" * 80)
print(f"\nTotal complaints in top 20: {total_complaints:,}")
print(f"Top 3 account for: {df_neighborhoods.head(3)['Percentage'].sum():.1f}% of top 20 total")
print("\n" + df_neighborhoods.to_string(index=False))

print("\n[4] Creating Comparison Chart: Boroughs vs Neighborhoods...")

# Separate boroughs and neighborhoods
df_boroughs = df_neighborhoods[df_neighborhoods['Type'] == 'Borough'].copy()
df_actual_neighborhoods = df_neighborhoods[df_neighborhoods['Type'] == 'Neighborhood'].head(10).copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), facecolor='white')

# LEFT: Boroughs
ax1.set_facecolor('white')
bars1 = ax1.barh(df_boroughs['City'], df_boroughs['Complaints'],
                 color=NEGATIVE_COLOR, edgecolor='#333333', linewidth=1.5)

for i, row in enumerate(df_boroughs.itertuples()):
    ax1.text(row.Complaints + 30000, i, f"{row.Complaints:,.0f}",
             va='center', ha='left', fontsize=11, fontweight='bold', color='#333333')

ax1.set_xlabel('Complaints', fontsize=12, fontweight='bold')
ax1.set_title('BOROUGHS (Geographic Aggregates)', fontsize=14, fontweight='bold', pad=15)
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000000)}M' if x >= 1000000 else f'{int(x/1000)}K'))
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(False)

# RIGHT: Actual Neighborhoods
ax2.set_facecolor('white')
df_actual_sorted = df_actual_neighborhoods.sort_values('Complaints', ascending=True)
bars2 = ax2.barh(df_actual_sorted['City'], df_actual_sorted['Complaints'],
                 color=PRIMARY_COLOR, edgecolor='#333333', linewidth=1.5)

for i, row in enumerate(df_actual_sorted.itertuples()):
    ax2.text(row.Complaints + 2000, i, f"{row.Complaints:,.0f}",
             va='center', ha='left', fontsize=11, fontweight='bold', color='#333333')

ax2.set_xlabel('Complaints', fontsize=12, fontweight='bold')
ax2.set_title('TOP 10 SPECIFIC NEIGHBORHOODS', fontsize=14, fontweight='bold', pad=15)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(False)

plt.suptitle('Borough Aggregates vs. Specific Neighborhood Hotspots',
             fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f'{output_dir}/static/top20_neighborhoods_comparison.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/top20_neighborhoods_comparison.svg', bbox_inches='tight')
print(f"✓ Saved: top20_neighborhoods_comparison.png/svg")
plt.close()

In [ ]:

print("\n[1] Creating Horizontal Bar Chart with Borough Highlight...")

fig, ax = plt.subplots(figsize=(14, 10), facecolor='white')
ax.set_facecolor('white')

# Sort data in descending order for top-down display
df_sorted = df_neighborhoods.sort_values('Complaints', ascending=True)

# Assign colors based on type
colors = [NEGATIVE_COLOR if t == 'Borough' else PRIMARY_COLOR
          for t in df_sorted['Type']]

# Create horizontal bars
bars = ax.barh(range(len(df_sorted)), df_sorted['Complaints'],
               color=colors, edgecolor='#333333', linewidth=1.2, height=0.7)

# Set y-axis labels
ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels(df_sorted['City'], fontsize=11)

# Add value labels with percentages
for i, row in enumerate(df_sorted.itertuples()):
    # Value label on the right
    ax.text(row.Complaints + 30000, i,
            f"{row.Complaints:,.0f}",
            va='center', ha='left', fontsize=10, fontweight='bold', color='#333333')

    # Percentage inside bar (if bar is wide enough)
    if row.Complaints > 100000:
        ax.text(row.Complaints * 0.5, i,
                f"{row.Percentage:.1f}%",
                va='center', ha='center', fontsize=10, fontweight='bold', color='white')

# Labels and title
ax.set_xlabel('Total Complaints', fontsize=14, fontweight='bold')
ax.set_title('Top 20 NYC Areas by Noise Complaints: Boroughs vs Neighborhoods',
             fontsize=18, fontweight='bold', pad=20)

# Format x-axis
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000000)}M' if x >= 1000000 else f'{int(x/1000)}K'))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=NEGATIVE_COLOR, edgecolor='#333333', label='Borough-Level'),
    Patch(facecolor=PRIMARY_COLOR, edgecolor='#333333', label='Specific Neighborhood')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11, frameon=True)

# Add annotation
ax.annotate('Manhattan (NEW YORK)\nand Brooklyn dominate,\naccounting for 62%\nof top 20 complaints',
            xy=(2157947, 19), xytext=(1500000, 15),
            arrowprops=dict(arrowstyle='->', color=NEGATIVE_COLOR, lw=2),
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='white',
                     edgecolor=NEGATIVE_COLOR, linewidth=2))

plt.tight_layout()
plt.savefig(f'{output_dir}/static/top20_neighborhoods_bar.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/top20_neighborhoods_bar.svg', bbox_inches='tight')
print(f"✓ Saved: top20_neighborhoods_bar.png/svg")
plt.close()

In [ ]:
"""
COMPLAINT TYPE DISTRIBUTION BY BOROUGH - VISUALIZATION
=======================================================
This code creates professional visualizations showing how complaint types
vary across NYC boroughs (percentage distribution).
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Professional neutral color palette
PRIMARY_COLOR = '#2C5F7C'      # Deep blue
SECONDARY_COLOR = '#5A9AB5'    # Medium blue
ACCENT_COLOR = '#E8A87C'       # Warm accent
NEGATIVE_COLOR = '#C1666B'     # Muted red
POSITIVE_COLOR = '#6B9080'     # Muted green
NEUTRAL_GRAY = '#6B7280'       # Professional gray

# Neutral palette for multiple categories
NEUTRAL_PALETTE = ['#2C5F7C', '#5A9AB5', '#6B9080', '#E8A87C', '#9B8E82',
                   '#C1666B', '#8B9DAF', '#A8DADC', '#457B9D', '#1D3557']

# Borough colors (consistent with previous visualizations)
BOROUGH_COLORS = {
    'MANHATTAN': '#2C5F7C',
    'BROOKLYN': '#5A9AB5',
    'QUEENS': '#6B9080',
    'BRONX': '#E8A87C',
    'STATEN ISLAND': '#9B8E82'
}

# Configure matplotlib for clean, white backgrounds
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0
plt.rcParams['axes.grid'] = False

DPI = 300
output_dir = 'nyc_noise_visualizations'

# Your data (percentages)
data = {
    'Borough': ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND'],
    'Commercial': [3.8, 10.9, 11.4, 9.1, 7.8],
    'Helicopter': [0.1, 1.3, 5.0, 2.8, 0.7],
    'Other': [2.9, 10.5, 16.0, 10.6, 14.9],
    'Park': [0.8, 1.1, 1.3, 1.3, 0.6],
    'Residential': [65.4, 52.0, 35.4, 53.8, 59.9],
    'Street/Sidewalk': [20.0, 17.3, 24.1, 12.9, 9.7],
    'Vehicle': [7.1, 6.9, 6.9, 9.6, 6.3]
}

df = pd.DataFrame(data)
df = df.set_index('Borough')

# Define borough order globally (used across all visualizations)
borough_order = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']

print("=" * 80)
print("COMPLAINT TYPE DISTRIBUTION BY BOROUGH - VISUALIZATION")
print("=" * 80)
print("\nData Preview (Percentages):")
print(df.to_string())
print("\n" + "=" * 80)


In [ ]:

print("\n[2] Creating Grouped Bar Chart...")

fig, ax = plt.subplots(figsize=(16, 8), facecolor='white')
ax.set_facecolor('white')

# Transpose for grouped bars
df_transposed = df.T

x = np.arange(len(df_transposed.index))
width = 0.15  # Width of bars

for i, borough in enumerate(borough_order):
    offset = width * (i - 2)  # Center the groups
    bars = ax.bar(x + offset, df_transposed[borough], width,
                  label=borough, color=BOROUGH_COLORS[borough],
                  edgecolor='#333333', linewidth=1)

# Formatting
ax.set_xlabel('Complaint Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage of Complaints (%)', fontsize=14, fontweight='bold')
ax.set_title('Residential Noise Dominates All Boroughs',
             fontsize=18, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(df_transposed.index, rotation=45, ha='right', fontsize=11)
ax.legend(title='Borough', fontsize=11, loc='upper right', frameon=True, title_fontsize=12)
ax.set_ylim(0, 70)

# Remove spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)

# Add horizontal line at 50% for reference
ax.axhline(y=50, color='#999999', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')

plt.tight_layout()
plt.savefig(f'{output_dir}/static/borough_complaint_distribution_grouped.png', dpi=DPI, bbox_inches='tight')
plt.savefig(f'{output_dir}/static/borough_complaint_distribution_grouped.svg', bbox_inches='tight')
print(f"✓ Saved: borough_complaint_distribution_grouped.png/svg")
plt.close()
